In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data/processed"
print(f"DATA_DIR: {DATA_DIR}")

In [ ]:
def load_data(start_date, end_date):
    spdata = pd.read_csv(str(DATA_DIR) + "/sandpdata.csv", index_col=0)
    spdata.index.name = "index"


    stockdata = pd.read_csv(
            str(DATA_DIR) + "/stock_date_top10_2023-04-30_2025-05-22.csv", index_col="Date"
        )
    stockdata.index = pd.to_datetime(stockdata.index)

    mask = (stockdata.index >= start_date) & (stockdata.index <= end_date)
    filtered_stockdata = stockdata[mask]
    return spdata, filtered_stockdata

In [ ]:
spdata, stockdata = load_data("2023-04-30", "2023-10-30")


In [ ]:
def run_PCA(data, n_components, normalize):
    """Run PCA on the given data.
    This function performs Principal Component Analysis (PCA) on the provided dataset.
    It allows for normalization of the data and specifies the number of principal components to retain.

    Args:
        data (dataframe): pandas DataFrame containing the data to be analyzed.
        n_components (int): Number of principal components to retain.
        normalize (boolean): Whether to normalize the data before applying PCA.

    Returns:
        _type_: A tuple containing the PCA object and the transformed data.
    """
    if "^GSPC" in data.columns:
        data_cleaned = data.drop(columns=["^GSPC"])
    else:
        data_cleaned = data.copy()

    returns = data_cleaned.pct_change().dropna()
    if normalize:
        scaler = StandardScaler()
        returns_scaled = scaler.fit_transform(returns)
    else:
        returns_scaled = returns
    pca = PCA(n_components=n_components)
    returns_pca = pca.fit_transform(returns_scaled)
    return returns, pca, returns_pca

In [ ]:
returns, pca, returns_pca = run_PCA(stockdata, n_components=0.8, normalize=True)

In [ ]:
pca.explained_variance_

In [ ]:
pca.components_ = pd.DataFrame(
    pca.components_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)],
    columns=returns.columns
)
pca.explained_variance_ratio_ = pd.Series(
    pca.explained_variance_ratio_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)
pca.explained_variance_ = pd.Series(
    pca.explained_variance_,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA on S&P 500 Stock Returns')
plt.grid(True)
plt.show()

In [ ]:
np.arange(1, int(len(pca.explained_variance_ratio_))+1)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(np.arange(1, int(len(pca.explained_variance_ratio_))+1), pca.explained_variance_ratio_, alpha=0.7, color='steelblue')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Each Principal Component')
plt.xticks(range(1, 31))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
loadings = pca.components_.T

In [ ]:
# Get loadings for PC1 to PC3
eig_port_weights = loadings[['PC1', 'PC2', 'PC3', 'PC4']]

# Normalize weights so each portfolio has unit sum (or any risk budget)
eig_port_weights = eig_port_weights.div(eig_port_weights.abs().sum(), axis=1)
# Multiply weights with actual (unscaled) returns
# `returns_new` is your original (not scaled) returns DataFrame
eig_port_returns = returns @ eig_port_weights  # matrix multiply

In [ ]:
eig_port_weights['PC2'].sort_values(ascending=False).head(10).plot(kind='bar', figsize=(10, 6), title='Top 10 Weights for PC1')

In [ ]:
eig_port_weights['PC2'].sort_values(ascending=False).tail(10).plot(kind='bar', figsize=(10, 6), title='Top 10 Weights for PC1')

In [ ]:
NumComponents = 4
eig_port_weights.plot.bar(subplots=True, layout=(int(NumComponents),1), figsize=(14,10), legend=False, sharey=True)


In [ ]:
# Step 5: Calculate Sharpe Ratios
risk_free_rate_daily = 0.05 / 252  # assuming 5% annual rate
# Subtract risk-free rate from each eigenportfolio return
excess_returns = eig_port_returns - risk_free_rate_daily
sharpe_ratios = excess_returns.mean() / eig_port_returns.std()
best_pc = sharpe_ratios.idxmax()
best_return_series = eig_port_returns[best_pc]

eigen_cum_return = (1 + best_return_series).cumprod()

In [ ]:
spy_return = stockdata['^GSPC'].pct_change().dropna()
spy_cum_return = (1 + spy_return).cumprod()

In [ ]:
# Plot
plt.figure(figsize=(12, 6))
plt.plot(spy_cum_return, label='S&P 500 (SPY)', alpha=0.8)
plt.plot(eigen_cum_return, label=f'Best Eigenportfolio ({best_pc})', linewidth=2)
plt.plot((1 + eig_port_returns['PC1']).cumprod(), label='PC0', linestyle='--', alpha=0.5)

plt.plot((1 + eig_port_returns['PC3']).cumprod(), label='PC3', linestyle='--', alpha=0.5)
plt.title('Cumulative Returns: S&P 500 vs Best Eigenportfolio')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
pca.components_.T

In [ ]:
columns = pca.components_.T.columns

In [ ]:
list(columns)

In [ ]:
sector_map = {spdata['Symbol'][i]: spdata['GICS Sector'][i] for i in range(len(spdata))}

In [ ]:
t_pca_components = pca.components_.T

In [ ]:
t_pca_components['Sector'] = t_pca_components.index.map(sector_map)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(data=t_pca_components, x='PC1', y='PC2', hue='Sector', palette='tab10', s=100)
plt.title('PCA Projection: Stocks Colored by Sector')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
spdata, future_stockdata = load_data("2024-03-01", "2024-06-01")

In [ ]:
future_returns = future_stockdata.pct_change().dropna().drop(columns=["^GSPC"], errors='ignore')
future_spreturn = future_stockdata["^GSPC"].pct_change().dropna()

In [ ]:
pc2_weights = loadings['PC1'] / loadings['PC1'].sum()

In [ ]:
portfolio_returns = future_returns.dot(pc2_weights)
cumulative_return = (1 + portfolio_returns).cumprod()

cumulative_return_sp = (1 + future_spreturn).cumprod()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(cumulative_return_sp, label='S&P 500 (SPY)', alpha=0.8)
plt.plot(cumulative_return, label=f'Best Eigenportfolio ({best_pc})', linewidth=2)
plt.title('Future Returns: S&P 500 vs Best Eigenportfolio')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np

# Sample PCA-transformed data
# Replace this with your actual PCA result
# Assume df_pca has columns: 'PC1', 'PC2', 'sector'
df_pca = pd.DataFrame({
    'PC1': np.random.randn(100),
    'PC2': np.random.randn(100),
    'sector': np.random.choice(['Tech', 'Finance', 'Health'], 100)
})

fig, ax = plt.subplots(figsize=(10, 6))

# Plot scatter
for sector, group in df_pca.groupby('sector'):
    ax.scatter(group['PC1'], group['PC2'], label=sector, alpha=0.6)

    # Compute ellipse parameters
    x_mean, y_mean = group['PC1'].mean(), group['PC2'].mean()
    cov = np.cov(group[['PC1', 'PC2']].T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    # Width and height = 2*sqrt(eigenvalue) to cover ~95% of the spread
    width, height = 2 * 2 * np.sqrt(eigvals)
    angle = np.degrees(np.arctan2(*eigvecs[:,0][::-1]))

    ellipse = patches.Ellipse((x_mean, y_mean), width, height,
                              edgecolor='black', facecolor='none', lw=1.5)
    ax.add_patch(ellipse)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.set_title('PCA Scatter Plot with Ellipses by Sector')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

returns = np.random.normal(0.001, 0.02, 1000)  # Simulated daily returns
portfolio_value = 1_000_000
confidence_level = 0.95

var_percentile = np.percentile(returns, (1 - confidence_level) * 100)
VaR = -var_percentile * portfolio_value
#print(f"1-day 95% VaR: ${VaR:.2f}")

In [ ]:
print(var_percentile)

In [ ]:
VaR

In [ ]:
np.percentile?

In [ ]:
np.percentile(np.random.normal(0,1,100), 5)